In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display
import types

%matplotlib inline

In [2]:
data_dir = Path("../docs/points_data")
files = sorted(data_dir.glob("*.geojson"))

# Set to None to use all files.
MAX_FILES = 1
selected_files = files if MAX_FILES is None else files[:MAX_FILES]

rows = []
for file in selected_files:
    with open(file) as f:
        geojson = json.load(f)

    for feature in geojson.get("features", []):
        geometry = feature.get("geometry", {})
        if geometry.get("type") != "Point":
            continue

        coords = geometry.get("coordinates", [])
        if len(coords) < 2:
            continue

        props = feature.get("properties", {})
        rows.append(
            {
                "GEO_LONG": coords[0],
                "GEO_LAT": coords[1],
                "date": props.get("date"),
                "location": props.get("location"),
                "event_type": props.get("event_type"),
                "category": props.get("category"),
                "goldstein": props.get("goldstein"),
                "tone": props.get("tone"),
                "source": props.get("source"),
                "url": props.get("url"),
            }
        )

gkg = pd.DataFrame(rows)

if gkg.empty:
    raise ValueError("No valid point features found in the selected GeoJSON files.")

gkg["date"] = pd.to_datetime(gkg["date"], format="%Y%m%d", errors="coerce")
gkg["goldstein"] = pd.to_numeric(gkg["goldstein"], errors="coerce")
gkg["tone"] = pd.to_numeric(gkg["tone"], errors="coerce")

gkg = gkg.dropna(subset=["GEO_LAT", "GEO_LONG", "event_type"]).reset_index(drop=True)

print(f"Loaded {len(gkg):,} events from {len(selected_files)} file(s).")
gkg.head()

Loaded 8,770 events from 1 file(s).


,GEO_LONG,GEO_LAT,date,location,event_type,category,goldstein,tone,source,url
0,-76.63110,36.9824,2024-01-02,"Smithfield, Virginia, United States",Public statement,diplomacy,0.4,-6.74,nbcbayarea.com,https://www.nbcbayarea.com/news/national-inter...
1,-84.38800,33.7490,2024-01-02,"Atlanta, Georgia, United States",Consultation,diplomacy,1.0,1.39,fbherald.com,https://www.fbherald.com/news/state/jimmy-and-...
2,-82.54990,40.7834,2024-12-02,"Richland County, Ohio, United States",Armed conflict,conflict,-10.0,-3.24,fox56.com,https://fox56.com/news/nation-world/1981-ohio-...
3,-0.11667,51.5000,2024-12-02,"London, London, City of, United Kingdom",Public statement,diplomacy,-0.4,-2.68,morningstaronline.co.uk,https://morningstaronline.co.uk/article/h-lead...
4,116.38800,39.9289,2024-12-25,"Beijing, Beijing, China",Coercion,pressure,-5.0,-5.14,nzherald.co.nz,https://www.nzherald.co.nz/business/xi-jinping...


In [3]:
categories = sorted(gkg["event_type"].dropna().unique())

cmap = plt.get_cmap("tab20", len(categories))

color_map = {cat: cmap(i) for i, cat in enumerate(categories)}

plotly_color_map = {k: mcolors.to_hex(v) for k, v in color_map.items()}


In [4]:
if "plotly_color_map" not in globals() and "color_map" in globals():
    plotly_color_map = {k: mcolors.to_hex(v) for k, v in color_map.items()}

if "categories" not in globals():
    categories = sorted(gkg["event_type"].dropna().unique())

gkg["date"] = pd.to_datetime(gkg["date"], errors="coerce")
df_all = gkg.dropna(subset=["GEO_LAT", "GEO_LONG", "event_type"]).copy()

if df_all.empty:
    raise ValueError("No valid rows found for map.")

POINT_ZOOM_THRESHOLD = 6.0
DENSITY_RADIUS = 10
DENSITY_ZMAX = 12
SHOW_COLORBAR = False

def _hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    if len(h) == 3:
        h = "".join([c * 2 for c in h])
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

hybrid_fig = go.FigureWidget()

for category in categories:
    sub = df_all[df_all["event_type"] == category]
    color = plotly_color_map.get(category, "#666666")

    colorscale = [
        [0.00, _hex_to_rgba(color, 0.00)],
        [0.03, _hex_to_rgba(color, 0.08)],
        [0.08, _hex_to_rgba(color, 0.18)],
        [0.18, _hex_to_rgba(color, 0.35)],
        [0.35, _hex_to_rgba(color, 0.55)],
        [0.60, _hex_to_rgba(color, 0.80)],
        [1.00, _hex_to_rgba(color, 1.00)],
    ]

    hybrid_fig.add_trace(
        go.Densitymap(
            lat=sub["GEO_LAT"].tolist(),
            lon=sub["GEO_LONG"].tolist(),
            z=[1] * len(sub),
            radius=DENSITY_RADIUS,
            colorscale=colorscale,
            zmin=0,
            zmax=DENSITY_ZMAX,
            zauto=False,
            showscale=SHOW_COLORBAR,
            hovertemplate=f"<b>{category}</b><extra></extra>",
            name=f"{category}__density",
            visible=True,
        )
    )

    hybrid_fig.add_trace(
        go.Scattermap(
            lat=sub["GEO_LAT"].tolist(),
            lon=sub["GEO_LONG"].tolist(),
            mode="markers",
            marker=dict(size=9, color=color, opacity=1.0),
            hovertext=sub["location"].fillna("").tolist(),
            hovertemplate="<b>%{customdata[2]}</b><br>%{hovertext}<br>%{customdata[0]|%Y-%m-%d}<extra></extra>",
            customdata=sub[
                ["date", "location", "event_type", "goldstein", "tone", "source", "url"]
            ].to_numpy(dtype=object),
            name=f"{category}__points",
            visible=False,
            showlegend=False,
            selected=dict(marker=dict(size=11, opacity=1.0)),
            unselected=dict(marker=dict(opacity=0.5)),
        )
    )

hybrid_fig.update_layout(
    autosize=True,
    width=None,
    height=700,
    dragmode="pan",
    clickmode="event+select",
    margin=dict(l=0, r=0, t=40, b=0),
    map=dict(style="carto-positron", center=dict(lat=20, lon=0), zoom=1),
    title="Global Events: Density + Points",
    showlegend=False,
)

PIN_DEFAULT_HTML = """
<div style="padding:10px;border:1px solid #ccc;border-radius:8px;">
  Click a point to pin event details here.
</div>
"""

info_box = widgets.HTML(value=PIN_DEFAULT_HTML)
pin_title = widgets.HTML("<b>Pinned Event</b>")
clear_pin_btn = widgets.Button(description="Clear pin", icon="times")

from html import escape

def _normalize_location(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()

def _current_pinned_df():
    pinned_df = _current_filtered_df()
    enabled_categories = [
        category for category, checkbox in checkboxes.items() if checkbox.value
    ]
    if enabled_categories:
        pinned_df = pinned_df[pinned_df["event_type"].isin(enabled_categories)]
    return pinned_df

def _rows_for_location(location_value):
    location_key = _normalize_location(location_value)
    if not location_key:
        return df_all.iloc[0:0].copy()

    pinned_df = _current_pinned_df()
    rows = pinned_df[pinned_df["location"].fillna("").astype(str).str.strip() == location_key].copy()
    if not rows.empty and "date" in rows.columns:
        rows = rows.sort_values(["date", "event_type"], na_position="last")
    return rows

def _render_pinned_rows(rows, location_value):
    if rows.empty:
        return PIN_DEFAULT_HTML

    location_text = escape(_normalize_location(location_value) or "N/A")
    cards = []

    for _, row in rows.iterrows():
        date_value = row.get("date")
        pretty_date = pd.to_datetime(date_value).strftime("%Y-%m-%d") if pd.notna(date_value) else "N/A"

        event_type = row.get("event_type")
        event_type_text = escape(str(event_type)) if pd.notna(event_type) else "N/A"
        source = row.get("source")
        url = row.get("url")
        goldstein = row.get("goldstein")
        tone = row.get("tone")
        location = row.get("location")

        safe_source = escape(str(source)) if pd.notna(source) else "N/A"
        if isinstance(url, str) and url:
            safe_link = f'<a href="{escape(url)}" target="_blank" rel="noopener noreferrer">{safe_source}</a>'
        else:
            safe_link = safe_source

        location_text_row = escape(str(location)) if pd.notna(location) else "N/A"
        cards.append(
            f"""
            <div style="padding:10px 12px;border:1px solid #ddd;border-radius:8px;background:#fff;margin-top:8px;">
              <div style="font-weight:700;margin-bottom:4px;">{event_type_text}</div>
              <div><b>Date:</b> {pretty_date}</div>
              <div><b>Location:</b> {location_text_row}</div>
              <div><b>Goldstein:</b> {goldstein}</div>
              <div><b>Tone:</b> {tone}</div>
              <div><b>Source:</b> {safe_link}</div>
            </div>
            """.strip()
        )

    return f"""
    <div style="padding:10px;border:1px solid #ccc;border-radius:8px;background:#fafafa;max-height:280px;overflow-y:auto;">
      <div style="font-weight:700;margin-bottom:6px;">Pinned Location: {location_text}</div>
      <div style="margin-bottom:4px;">{len(rows):,} article(s) at this location</div>
      {''.join(cards)}
    </div>
    """

def _set_info_from_point(trace, point_index):
    custom = trace.customdata
    if custom is None or point_index >= len(custom):
        return

    row = custom[point_index]
    if hasattr(row, "tolist"):
        row = row.tolist()
    if not isinstance(row, (list, tuple)):
        row = [row]

    location = row[1] if len(row) > 1 else None
    matched_rows = _rows_for_location(location)
    if matched_rows.empty:
        info_box.value = PIN_DEFAULT_HTML
    else:
        info_box.value = _render_pinned_rows(matched_rows, location)

def _clear_pin(_):
    info_box.value = PIN_DEFAULT_HTML

clear_pin_btn.on_click(_clear_pin)

def _handle_click(trace, points, state):
    if points.point_inds:
        _set_info_from_point(trace, points.point_inds[0])

def _handle_selection(trace, points, selector):
    if points.point_inds:
        _set_info_from_point(trace, points.point_inds[-1])

for tr in hybrid_fig.data:
    if tr.name.endswith("__points"):
        tr.on_click(_handle_click)
        tr.on_selection(_handle_selection)

trace_pairs = {category: {"density": None, "points": None} for category in categories}

for tr in hybrid_fig.data:
    if tr.name.endswith("__density"):
        trace_pairs[tr.name[:-9]]["density"] = tr
    elif tr.name.endswith("__points"):
        trace_pairs[tr.name[:-8]]["points"] = tr

checkboxes = {}
legend_rows = []

for category in categories:
    cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="20px"))
    checkboxes[category] = cb

    color = plotly_color_map.get(category, "#666666")
    swatch = widgets.HTML(
        value=f'<span style="width:12px;height:12px;border-radius:50%;background:{color};display:inline-block;"></span>',
        layout=widgets.Layout(width="16px"),
    )
    label = widgets.HTML(value=f'<span style="font-size:12px;line-height:1.2;">{category}</span>')
    legend_rows.append(
        widgets.HBox([cb, swatch, label], layout=widgets.Layout(align_items="center", gap="8px"))
    )

date_values = sorted(df_all["date"].dropna().dt.normalize().unique())
if not date_values:
    raise ValueError("No valid dates available for date range slider.")

date_options = [(d.strftime("%d.%m.%Y"), d) for d in date_values]
date_range = widgets.SelectionRangeSlider(
    options=date_options,
    value=(date_values[0], date_values[-1]),
    description="Date:",
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
    style={"description_width": "40px"},
)

mode_toggle = widgets.ToggleButtons(
    options=[("Auto", "auto"), ("Density", "density"), ("Points", "points")],
    value="auto",
    description="View:",
    style={"description_width": "40px"},
)

radius_slider = widgets.IntSlider(
    value=DENSITY_RADIUS,
    min=1,
    max=150,
    step=1,
    description="Radius:",
    continuous_update=False,
    style={"description_width": "40px"},
    layout=widgets.Layout(width="280px"),
)

refresh_btn = widgets.Button(description="Refresh", icon="refresh")

_zoom_state = {"value": None}

def _extract_zoom(relayout):
    if not isinstance(relayout, dict):
        return None
    if "map.zoom" in relayout:
        try:
            return float(relayout["map.zoom"])
        except Exception:
            pass
    derived = relayout.get("map._derived")
    if isinstance(derived, dict) and "zoom" in derived:
        try:
            return float(derived["zoom"])
        except Exception:
            pass
    return None

def _current_filtered_df():
    start, end = date_range.value
    return df_all[(df_all["date"] >= start) & (df_all["date"] <= end)]

def _get_zoom():
    # Prefer live layout zoom if available, then fallback to tracked relayout zoom.
    try:
        live_zoom = hybrid_fig.layout.map.zoom
        if live_zoom is not None:
            _zoom_state["value"] = float(live_zoom)
            return float(live_zoom)
    except Exception:
        pass

    zoom = _zoom_state.get("value")
    if zoom is None:
        zoom = 1.0
    return float(zoom)

def _should_show_points():
    mode = mode_toggle.value
    if mode == "points":
        return True
    if mode == "density":
        return False
    return _get_zoom() >= POINT_ZOOM_THRESHOLD

def _refresh_layers(_=None):
    dff = _current_filtered_df()
    show_points = _should_show_points()
    current_radius = int(radius_slider.value)

    with hybrid_fig.batch_update():
        for category in categories:
            pair = trace_pairs.get(category, {})
            dtr = pair.get("density")
            ptr = pair.get("points")
            if dtr is None or ptr is None:
                continue

            sub = dff[dff["event_type"] == category]
            enabled = bool(checkboxes[category].value)
            has_data = len(sub) > 0

            lats = sub["GEO_LAT"].tolist()
            lons = sub["GEO_LONG"].tolist()

            dtr.lat = lats
            dtr.lon = lons
            dtr.z = [1] * len(sub)
            dtr.radius = current_radius

            ptr.lat = lats
            ptr.lon = lons
            ptr.hovertext = sub["location"].fillna("").tolist()
            ptr.customdata = sub[
                ["date", "location", "event_type", "goldstein", "tone", "source", "url"]
            ].to_numpy(dtype=object)

            dtr.visible = enabled and has_data and (not show_points)
            ptr.visible = enabled and has_data and show_points

        start, end = date_range.value
        if mode_toggle.value == "auto":
            mode_text = "Points" if show_points else "Density"
        elif mode_toggle.value == "points":
            mode_text = "Points (forced)"
        else:
            mode_text = "Density (forced)"

        hybrid_fig.layout.title = (
            f"Global Events: {mode_text} ({len(dff):,} points, radius={current_radius}) | "
            f"{start:%d.%m.%Y} to {end:%d.%m.%Y}"
        )

_orig_relayout_handler = hybrid_fig._handler_js2py_relayout

def _safe_relayout_handler(self, change):
    relayout = change.get("new")
    derived_zoom = _extract_zoom(relayout)
    if derived_zoom is not None:
        _zoom_state["value"] = derived_zoom

    if isinstance(relayout, dict):
        filtered = {
            k: v
            for k, v in relayout.items()
            if not (k.endswith("._derived") or "._derived[" in k)
        }
        if filtered != relayout:
            change = dict(change)
            change["new"] = filtered
        relayout = filtered

    try:
        out = _orig_relayout_handler(change)
    except ValueError as exc:
        if "Invalid property path" in str(exc):
            out = None
        else:
            raise

    # In auto mode, refresh after any relayout because some backends don't emit map.zoom reliably.
    if mode_toggle.value == "auto":
        _refresh_layers()

    return out

hybrid_fig._handler_js2py_relayout = types.MethodType(_safe_relayout_handler, hybrid_fig)

def _native_relayout_cb(*args):
    relayout = None
    for arg in reversed(args):
        if isinstance(arg, dict):
            relayout = arg
            break

    z = _extract_zoom(relayout)
    if z is not None:
        _zoom_state["value"] = z

    if mode_toggle.value == "auto":
        _refresh_layers()

if hasattr(hybrid_fig, "on_relayout"):
    hybrid_fig.on_relayout(_native_relayout_cb)

for cb in checkboxes.values():
    cb.observe(_refresh_layers, names="value")

date_range.observe(_refresh_layers, names="value")
mode_toggle.observe(_refresh_layers, names="value")
radius_slider.observe(_refresh_layers, names="value")

def _refresh_button_clicked(_):
    # Force a live zoom read before recalculating auto visibility.
    try:
        z = hybrid_fig.layout.map.zoom
        if z is not None:
            _zoom_state["value"] = float(z)
    except Exception:
        pass
    _refresh_layers()

refresh_btn.on_click(_refresh_button_clicked)

legend_box = widgets.VBox(
    [widgets.HTML("<b>event_type</b>")] + legend_rows,
    layout=widgets.Layout(
        padding="10px 12px",
        border="1px solid #ccc",
        border_radius="8px",
        background_color="white",
        min_width="260px",
        max_width="320px",
        overflow_y="auto",
        max_height="760px",
    ),
)

controls = widgets.VBox(
    [date_range, widgets.HBox([mode_toggle, radius_slider, refresh_btn])],
    layout=widgets.Layout(
        width="100%",
        padding="8px 12px",
        border="1px solid #ccc",
        border_radius="8px",
        background_color="white",
        margin="0 0 8px 0",
    ),
)

pin_controls = widgets.HBox(
    [pin_title, clear_pin_btn],
    layout=widgets.Layout(justify_content="space-between", align_items="center", width="100%"),
)

map_panel = widgets.VBox(
    [controls, hybrid_fig, pin_controls, info_box],
    layout=widgets.Layout(flex="1 1 auto", width="100%"),
)

_refresh_layers()

display(
    widgets.HBox(
        [map_panel, legend_box],
        layout=widgets.Layout(width="100%", align_items="flex-start"),
    )
)